In [ ]:
import numpy as np 
import pandas as pd
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Conv2D , MaxPooling2D , Flatten , BatchNormalization , Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import cv2 

#    Generators & Loading Data

In [ ]:

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    validation_split=0.2  # النسبة اللي هيتم سحبها للاختبار
)

# مسار فولدر التدريب الأساسي (اللي جواه فولدرين cats و dogs)
train_dir = r'F:\Documents\ML & AI\Projects\data\Cats & Dogs\train'

# (2. سحب بيانات التدريب (80%
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='training'  # تحديد إن ده جزء التدريب
)

# 3. سحب بيانات الاختبار (20%) من نفس الفولدر
validation_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='validation'  # تحديد إن ده جزء الاختبار
)

#   CNN Model

In [ ]:

model = Sequential([
    # ==========================================
    # الكتلة الأولى: استخراج الملامح البسيطة (زي الحواف والخطوط)
    # ==========================================
    Conv2D(32,                           # عدد الفلاتر اللي هتمسح الصورة (32 فلتر)
           kernel_size=(3,3),            # حجم الفلتر نفسه (مربع 3 بيكسل في 3 بيكسل)
           padding='valid',              # عدم إضافة حواف وهمية (أصفار) حول الصورة
           activation='relu',            # تحويل الأرقام السالبة لصفر عشان الموديل يفهم الأنماط المعقدة
           input_shape=(150,150,3)),     # أبعاد الصورة اللي داخله (الطول، العرض، 3 قنوات للألوان).. بنكتبها في أول طبقة بس
    
    BatchNormalization(),                # تظهير وتوحيد الأرقام عشان نسرع عملية التدريب ونمنع التشتت
    
    MaxPooling2D(pool_size=(2,2),        # استخراج أهم (أكبر) بيكسل من كل مربع 2x2
                 strides=2,              # تحريك الفلتر خطوتين بخطوتين
                 padding='valid'),       # ده بيؤدي لتصغير أبعاد الصورة للنص بالظبط لتقليل الحسابات
                 
    # ==========================================
    # الكتلة التانية: استخراج ملامح أعقد (زي شكل الأذن أو العين)
    # ==========================================
    Conv2D(64,                           # زودنا عدد الفلاتر لـ 64 عشان نركز على تفاصيل أكتر
           kernel_size=(3,3), 
           padding='valid', 
           activation='relu'),
           
    BatchNormalization(),                # تسريع التدريب زي ما عملنا فوق
    
    MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'), # تصغير الصورة للنص مرة كمان
    
    # ==========================================
    # الكتلة التالتة: استخراج الملامح الدقيقة جداً والمترابطة
    # ==========================================
    Conv2D(128,                          # 128 فلتر للتفاصيل الدقيقة
           kernel_size=(3,3), 
           padding='valid', 
           activation='relu'),
           
    BatchNormalization(),
    
    MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'),

    # ==========================================
    # المرحلة الانتقالية
    # ==========================================
    Flatten(),                           # فرد المصفوفات والمربعات دي كلها لخط واحد طويل (1D)

    # ==========================================
    # مرحلة اتخاذ القرار (Fully Connected)
    # ==========================================
    Dense(128, activation='relu'),       # طبقة عصبية فيها 128 خلية عشان تربط الملامح ببعضها وتاخد قرار مبدئي
    
    Dropout(0.1),                        # إطفاء 10% من الخلايا عشوائياً عشان نمنع الموديل من إنه يحفظ (Overfitting)

    Dense(64, activation='relu'),        # طبقة عصبية تانية فيها 64 خلية لتأكيد القرار
    
    Dropout(0.1),                        # إطفاء 10% تانيين للحماية

    # ==========================================
    # الطبقة النهائية (المخرجات)
    # ==========================================
    Dense(1,                             # خلية واحدة بس لأننا بنصنف حاجة من اتنين (تصنيف ثنائي)
          activation='sigmoid')          # بتضغط الأرقام وتطلعلك نسبة مئوية (من 0 لـ 1).. أقرب لـ 1 يعني كلب، أقرب لـ 0 يعني قطة
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
history = model.fit(
    train_generator,                     # الداتا اللي هيذاكر منها (صور التدريب)
    epochs=15,                           # هيذاكر الداتا دي كلها كام مرة؟
    validation_data=validation_generator # الداتا اللي هيمتحن فيها بعد كل لفة عشان يقيس مستواه
)

In [ ]:
plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='test')
plt.legend()
plt.show()

In [ ]:
import cv2 
import matplotlib.pyplot as plt

test_image = cv2.imread(r'F:\Documents\ML & AI\Projects\data\A-Dog.jpg')
test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
plt.imshow(test_image)

In [ ]:
test_image.shape

In [ ]:
test_image_resized = cv2.resize(test_image, (150, 150))

In [ ]:
test_image_batched = np.expand_dims(test_image_resized, axis=0)

In [ ]:
prediction = model.predict(test_image_batched)

In [ ]:
if prediction[0][0] > 0.5:
    print("it is a dog")
else:
    print("it is a cat")